In [3]:
import os

os.environ["PYSPARK_PYTHON"] = ".venv/bin/python"
os.environ["PYSPARK_DRIVER_PYTHON"] = ".venv/bin/python"
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("iceberg-maintenance-experiments")
    .config("spark.jars.packages", ",".join([
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.7.0",
        "org.apache.iceberg:iceberg-aws-bundle:1.7.0",
        "org.apache.hadoop:hadoop-aws:3.3.4",
    ]))
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.glue", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.glue.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
    .config("spark.sql.catalog.glue.warehouse", "s3://binance-iceberg-lake/warehouse")
    .config("spark.sql.catalog.glue.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.parquet.enableVectorizedReader", "false")
    .config("spark.sql.iceberg.vectorization.enabled", "false")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "3g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

:: loading settings :: url = jar:file:/home/ubuntu/binance-iceberg-lakehouse/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ubuntu/.ivy2/cache
The jars for the packages stored in: /home/ubuntu/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ea248f58-1eef-43d2-9581-8a6c09c30eb7;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.7.0 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.7.0 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 282ms :: artifacts dl 10ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.apache.iceberg#iceberg-aws-bundle;1.7.0 from cen

In [ ]:
from pathlib import Path
import pandas as pd
import time

RESULT_DIR = Path("results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

def sql_df(query: str):
    return spark.sql(query)

def show_pd(query: str):
    return spark.sql(query).toPandas()

def timed_sql(query: str):
    start = time.time()
    df = spark.sql(query)
    rows = df.collect()
    elapsed = time.time() - start
    return rows, elapsed

def save_markdown(df: pd.DataFrame, path: str, title: str):
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with out.open("w", encoding="utf-8") as f:
        f.write(f"# {title}\n\n")
        f.write(df.to_markdown(index=False))
        f.write("\n")

# Compaction File Stats Experiment

This experiment compares Iceberg file statistics before and after `rewrite_data_files`.
Target table: `glue.binance_lakehouse.market_hourly_summary`

In [5]:
target = "glue.binance_lakehouse.market_hourly_summary"

before_files = show_pd(f"""
SELECT
  COUNT(*) AS file_count,
  ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 4) AS avg_file_size_mb,
  ROUND(SUM(file_size_in_bytes) / 1024 / 1024, 4) AS total_size_mb,
  SUM(record_count) AS total_records
FROM {target}.files
""")

before_files

,file_count,avg_file_size_mb,total_size_mb,total_records
0,31,0.0068,0.2109,744


In [8]:
# Query time before, twice for cache
query = f"""
SELECT symbol, COUNT(*) AS hour_count, SUM(trade_count) AS total_trades
FROM {target}
GROUP BY symbol
"""

_, before_elapsed = timed_sql(query)
before_elapsed

2.396036148071289

In [10]:
spark.sql("""
CALL glue.system.rewrite_data_files(
  table => 'binance_lakehouse.market_hourly_summary',
  options => map('min-input-files','2')
)
""").show(truncate=False)
after_files = show_pd(f"""
SELECT
  COUNT(*) AS file_count,
  ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 4) AS avg_file_size_mb,
  ROUND(SUM(file_size_in_bytes) / 1024 / 1024, 4) AS total_size_mb,
  SUM(record_count) AS total_records
FROM {target}.files
""")

after_files
_, after_elapsed = timed_sql(query)
after_elapsed

+--------------------------+----------------------+---------------------+-----------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|
+--------------------------+----------------------+---------------------+-----------------------+
|0                         |0                     |0                    |0                      |
+--------------------------+----------------------+---------------------+-----------------------+



2.0678603649139404

In [ ]:
result = pd.DataFrame([
    {"stage": "before", **before_files.iloc[0].to_dict(), "query_time_sec": before_elapsed},
    {"stage": "after", **after_files.iloc[0].to_dict(), "query_time_sec": after_elapsed},
])

save_markdown(
    result,
    "experiments/results/compaction_file_stats.md",
    "Compaction File Stats Result"
)

result

,stage,file_count,avg_file_size_mb,total_size_mb,total_records,query_time_sec
0,before,31.0,0.0068,0.2109,744.0,2.396036
1,after,31.0,0.0068,0.2109,744.0,2.067860
